# Fragrantica — detalhes + reviews (1 visita por perfume)

Lê o `perfumes.csv`, abre cada perfume **uma vez** e grava:

- `reviews.csv` — 1 linha por review: `id_perfume, designer, perfume, review_id, ordem, autor, data_texto, data_aprox, tags, texto`
- `detalhes_perfumes.csv` — 1 linha por perfume: gênero, ano, nota média, votos, nº de resenhas, perfumistas, acordes, notas de topo/coração/base
- `reviews_controle.csv` — perfumes já visitados (inclusive sem review), para retomar de onde parou

Se um perfume tem resenhas no site mas nenhuma foi capturada, salva **print + HTML** em `debug_reviews/`.

In [ ]:
# ---------- 1. Instalação (uma vez por sessão) ----------
!wget -q -nc https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get update -qq
!apt-get install -y -qq ./google-chrome-stable_current_amd64.deb xvfb
!pip install -q -U seleniumbase pyvirtualdisplay
!pkill -f chrome || true

In [ ]:
# ---------- 2. Drive ----------
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ---------- 3. Imports ----------
import os
import re
import time
import random
import pandas as pd
from tqdm.auto import tqdm
from seleniumbase import Driver
from pyvirtualdisplay import Display
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

In [ ]:
# ---------- 4. Configurações ----------
MODO_TESTE = True           # True = só os PERFUMES_TESTE, começando do zero
PERFUMES_TESTE = ["Sauvage", "Bleu de Chanel", "Khamrah"]

PASTA = "/content/drive/MyDrive/"          # pasta onde está o perfumes.csv
ARQUIVO_ENTRADA = PASTA + "perfumes.csv"
SUFIXO = "_teste" if MODO_TESTE else ""
ARQUIVO_REVIEWS = PASTA + f"reviews{SUFIXO}.csv"
ARQUIVO_CONTROLE = PASTA + f"reviews_controle{SUFIXO}.csv"
ARQUIVO_DETALHES = PASTA + f"detalhes_perfumes{SUFIXO}.csv"
PASTA_DEBUG = PASTA + "debug_reviews/"
os.makedirs(PASTA_DEBUG, exist_ok=True)

LIMITE_POR_DESIGNER = None  # ex.: 30 = só os 30 primeiros perfumes de cada marca
MAX_POR_SESSAO = 2000
MAX_CLIQUES_REVIEWS = 100   # cliques em "mais resenhas" por perfume (cada clique ~7 reviews)
ESPERA_REVIEWS = 12         # segundos esperando as resenhas aparecerem
TAMANHO_LOTE = 25
REINICIAR_DRIVER_A_CADA = 300
RECONNECT = 3
PAUSA = (1.5, 3.5)

if MODO_TESTE:
    for f in (ARQUIVO_REVIEWS, ARQUIVO_CONTROLE, ARQUIVO_DETALHES):
        if os.path.exists(f):
            os.remove(f)

In [ ]:
# ---------- 5. Driver ----------
display = None

def criar_driver():
    global display
    if display is None:
        display = Display(visible=0, size=(1920, 1080))
        display.start()
    driver = Driver(
        uc=True,
        binary_location="/usr/bin/google-chrome",
        page_load_strategy="eager",
        block_images=True,
    )
    driver.set_page_load_timeout(30)
    return driver


def fechar(driver):
    global display
    try:
        driver.quit()
    except Exception:
        pass
    if display is not None:
        display.stop()
        display = None


def abrir(driver, url, tentativas=3):
    for t in range(1, tentativas + 1):
        try:
            driver.uc_open_with_reconnect(url, RECONNECT)
        except TimeoutException:
            driver.execute_script("window.stop();")
        try:
            WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.TAG_NAME, "h1")))
        except TimeoutException:
            pass

        titulo = driver.title.lower()
        if "just a moment" in titulo or "um momento" in titulo:
            print(f"⚠ Cloudflare (tentativa {t}), aguardando...")
            time.sleep(15 * t)
            continue
        if driver.find_elements(By.TAG_NAME, "h1"):
            return
        time.sleep(5 * t)
    raise RuntimeError(f"Não carregou (título: {driver.title!r})")

In [ ]:
# ---------- 6a. Detalhes do perfume ----------
# JavaScript que roda dentro da página e devolve tudo de uma vez (bem mais rápido
# do que vários find_elements).
JS_EXTRAIR = r'''
const txt = el => (el ? el.textContent.replace(/\s+/g, ' ').trim() : '');
const h1 = txt(document.querySelector('h1'));

// Acordes principais: links para /accords-search/
const acordes = [...new Set(
  [...document.querySelectorAll('a[href*="/accords-search/"]')].map(txt).filter(Boolean)
)];

// Perfumistas: links para /narizes/
const perfumistas = [...new Set(
  [...document.querySelectorAll('a[href*="/narizes/"]')].map(txt).filter(Boolean)
)];

// Pirâmide: percorre a página em ordem; cada título "Notas de ..." define o nível
// e os links /notas/ seguintes entram nesse nível.
const niveis = {topo: [], coracao: [], base: [], gerais: []};
let nivel = null;
const walker = document.createTreeWalker(document.body, NodeFilter.SHOW_ELEMENT);
while (walker.nextNode()) {
  const el = walker.currentNode;
  if (el.children.length === 0 || /^H[1-6]$/.test(el.tagName)) {
    const t = txt(el).toLowerCase();
    if (/^notas de topo$/.test(t)) nivel = 'topo';
    else if (/^notas de cora[cç][aã]o$/.test(t)) nivel = 'coracao';
    else if (/^notas de (base|fundo)$/.test(t)) nivel = 'base';
  }
  if (el.tagName === 'A' && (el.getAttribute('href') || '').includes('/notas/')) {
    const nome = txt(el);
    if (!nome) continue;
    const alvo = niveis[nivel || 'gerais'];
    if (!alvo.includes(nome)) alvo.push(nome);
  }
}

return {
  h1: h1,
  corpo: document.body.innerText,
  acordes: acordes,
  perfumistas: perfumistas,
  notas: niveis,
};
'''


def numero(s):
    return int(re.sub(r"\D", "", s)) if s else None


def extrair_detalhes(driver, row):
    """Lê os dados do perfume da página já aberta."""
    d = driver.execute_script(JS_EXTRAIR)
    corpo = d["corpo"]
    titulo_pag = driver.title  # ex.: "Abaan Lattafa Perfumes perfume - a fragrância Feminino 2022"

    nota = re.search(r"([\d]+[.,]\d+)\s*fora de\s*5", corpo)
    votos = re.search(r"([\d.,]+)\s*votos", corpo)
    reviews = re.search(r"resenhas\s*\(\s*([\d.,]+)\s*\)", corpo, re.I)
    ano = (re.search(r"lan[çc]ad[oa]\s+em\s+(\d{4})", corpo, re.I)
           or re.search(r"(\d{4})\s*$", titulo_pag))

    h1 = (d["h1"] + " " + titulo_pag).lower()
    if "para homens e mulheres" in h1 or "compartilh" in h1 or "unissex" in h1:
        genero = "compartilhável"
    elif "para mulheres" in h1 or "feminino" in h1:
        genero = "feminino"
    elif "para homens" in h1 or "masculino" in h1:
        genero = "masculino"
    else:
        genero = None

    n = d["notas"]
    return {
        "id_perfume": row.id_perfume,
        "designer": row.designer,
        "perfume": row.perfume,
        "titulo": d["h1"],
        "genero": genero,
        "ano": int(ano.group(1)) if ano else None,
        "nota_media": float(nota.group(1).replace(",", ".")) if nota else None,
        "votos": numero(votos.group(1)) if votos else None,
        "reviews": numero(reviews.group(1)) if reviews else None,
        "perfumistas": "; ".join(d["perfumistas"]),
        "acordes": "; ".join(d["acordes"]),
        "notas_topo": "; ".join(n["topo"]),
        "notas_coracao": "; ".join(n["coracao"]),
        "notas_base": "; ".join(n["base"]),
        "notas_gerais": "; ".join(n["gerais"]),
        "link_perfume": row.link_perfume,
    }

In [ ]:
# ---------- 6b. Reviews ----------
JS_REVIEWS = r'''
const limpa = s => (s || '').replace(/\s+/g, ' ').trim();
const reData = /(\b(\d+|an?)\s+\w+\s+ago|h[áa]\s+(\d+|um|uma)\s+\w+|\d+\s+\w+\s+atr[áa]s|\d{1,2}\/\d{1,2}\/\d{2,4})/i;
const ignorar = /^(compartilhar|share|leia mais|ler mais|read more|mostrar mais|mostrar menos|responder|denunciar)$/i;
const out = [];
for (const corpo of document.querySelectorAll('div[id^="review_"]')) {
  // sobe até o maior bloco que ainda contém só esta resenha (o "card")
  let card = corpo;
  while (card.parentElement && card.parentElement !== document.body &&
         card.parentElement.querySelectorAll('div[id^="review_"]').length === 1) {
    card = card.parentElement;
  }
  const paragrafos = [...corpo.querySelectorAll('p')].map(p => limpa(p.textContent)).filter(Boolean);
  const texto = paragrafos.length ? paragrafos.join('\n') : limpa(corpo.textContent);
  // folhas de texto do card que estão fora do corpo da resenha
  const folhas = [...card.querySelectorAll('*')]
    .filter(e => !corpo.contains(e) &&
                 [...e.childNodes].some(n => n.nodeType === 3 && n.textContent.trim()))
    .map(e => limpa([...e.childNodes].filter(n => n.nodeType === 3).map(n => n.textContent).join(' ')))
    .filter(Boolean);
  const links = [...card.querySelectorAll('a')].filter(a => !corpo.contains(a))
    .map(a => limpa(a.textContent)).filter(t => t && !ignorar.test(t));
  const autor = links[0] || folhas.find(t => !reData.test(t) && !ignorar.test(t)) || '';
  const data = folhas.find(t => reData.test(t)) || '';
  const tags = folhas.filter(t => t !== autor && t !== data && t.length <= 30 && !ignorar.test(t));
  out.push({
    review_id: corpo.id.replace('review_', ''),
    autor: autor,
    data_texto: data,
    tags: [...new Set(tags)].join('; '),
    texto: texto,
  });
}
return out;
'''

JS_BOTAO_MAIS = r'''
const alvos = [...document.querySelectorAll('button, a, span, div')].filter(el => {
  if (el.children.length > 2 || el.offsetParent === null) return false;
  const t = (el.textContent || '').trim().toLowerCase();
  if (t.length > 40) return false;
  const href = el.getAttribute('href');
  if (href && !href.startsWith('#') && !href.startsWith('javascript')) return false;
  return /(mais|todas as|ver) (resenhas|reviews)|^(mostrar|carregar|ver) mais$|^(load|show) more/.test(t);
});
if (!alvos.length) return false;
alvos[alvos.length - 1].scrollIntoView({block: 'center'});
alvos[alvos.length - 1].click();
return true;
'''


UNIDADES = {
    "minute": "minutes", "minuto": "minutes", "hour": "hours", "hora": "hours",
    "day": "days", "dia": "days", "week": "weeks", "semana": "weeks",
    "month": "days30", "m[eê]s": "days30", "year": "days365", "ano": "days365",
}

def data_aproximada(txt, agora):
    # "19 hours ago", "há 3 dias", "2 years ago" -> data aproximada; datas absolutas também funcionam
    m = re.search(r"\b(\d+|an?|um|uma)\s+([a-zêç]+)", txt or "", re.I)
    if m:
        n, unidade = (int(m.group(1)) if m.group(1).isdigit() else 1), m.group(2).lower()
        for chave, tipo in UNIDADES.items():
            if re.match(chave, unidade):
                if tipo.startswith("days") and tipo != "days":
                    return agora - pd.Timedelta(days=n * int(tipo[4:]))
                return agora - pd.Timedelta(**{tipo: n})
    return pd.to_datetime(txt, errors="coerce", dayfirst=False) if txt else pd.NaT


CONTAR = "return document.querySelectorAll('div[id^=review_]').length"


def total_no_site(driver):
    corpo = driver.execute_script("return document.body.innerText")
    m = re.search(r"resenhas\s*\(\s*([\d.,]+)\s*\)", corpo, re.I)
    return int(re.sub(r"\D", "", m.group(1))) if m else None


def esperar_reviews(driver):
    """Rola a página aos poucos (dispara o carregamento preguiçoso) até as resenhas aparecerem."""
    driver.execute_script(
        "const a = document.querySelector('#all-reviews, [id*=review], [name=all-reviews]');"
        "if (a) a.scrollIntoView({block: 'start'});")
    fim = time.time() + ESPERA_REVIEWS
    while time.time() < fim:
        if driver.execute_script(CONTAR):
            return True
        driver.execute_script("window.scrollBy(0, 700);")
        time.sleep(0.6)
    return driver.execute_script(CONTAR) > 0


def salvar_debug(driver, row):
    nome = f"{PASTA_DEBUG}{row.id_perfume}"
    driver.save_screenshot(nome + ".png")
    with open(nome + ".html", "w", encoding="utf-8") as f:
        f.write(driver.page_source)


def coletar_perfume(driver, row):
    """Abre a página uma vez: lê os detalhes (topo da página) e depois as reviews."""
    abrir(driver, row.link_perfume)
    detalhes = extrair_detalhes(driver, row)
    total = total_no_site(driver)
    esperar_reviews(driver)

    anteriores = -1
    for _ in range(MAX_CLIQUES_REVIEWS):
        qtd = driver.execute_script(CONTAR)
        if qtd == anteriores or not driver.execute_script(JS_BOTAO_MAIS):
            break
        anteriores = qtd
        time.sleep(random.uniform(1.2, 2.0))

    agora = pd.Timestamp.now().floor("min")
    registros = []
    for ordem, r in enumerate(driver.execute_script(JS_REVIEWS), start=1):
        registros.append({
            "id_perfume": row.id_perfume,
            "designer": row.designer,
            "perfume": row.perfume,
            "review_id": int(r["review_id"]) if r["review_id"].isdigit() else r["review_id"],
            "ordem": ordem,
            "autor": r["autor"],
            "data_texto": r["data_texto"],
            "data_aprox": data_aproximada(r["data_texto"], agora),
            "tags": r["tags"],
            "texto": r["texto"],
        })

    if total and not registros:
        salvar_debug(driver, row)
    detalhes["reviews_coletadas"] = len(registros)
    return detalhes, registros, total

In [ ]:
# ---------- 7. Fila (retoma de onde parou) ----------
df_links = pd.read_csv(ARQUIVO_ENTRADA).drop_duplicates("id_perfume")
if LIMITE_POR_DESIGNER:
    df_links = df_links.groupby("designer", sort=False).head(LIMITE_POR_DESIGNER)

feitos = set()
if os.path.exists(ARQUIVO_CONTROLE):
    feitos = set(pd.read_csv(ARQUIVO_CONTROLE, usecols=["id_perfume"])["id_perfume"])

fila = df_links[~df_links["id_perfume"].isin(feitos)]
if MODO_TESTE:
    escolhidos = fila[fila["perfume"].isin(PERFUMES_TESTE)]
    fila = escolhidos if len(escolhidos) else fila.head(3)
else:
    fila = fila.head(MAX_POR_SESSAO)

print(f"Perfumes: {len(df_links)} | já visitados: {len(feitos)} | nesta execução: {len(fila)}")

In [ ]:
# ---------- 8. Execução ----------
def salvar(registros, caminho):
    if registros:
        pd.DataFrame(registros).to_csv(caminho, mode="a", header=not os.path.exists(caminho), index=False)

def salvar_tudo():
    salvar(lote_reviews, ARQUIVO_REVIEWS)   # reviews e detalhes antes do controle
    salvar(lote_detalhes, ARQUIVO_DETALHES)
    salvar(lote_controle, ARQUIVO_CONTROLE)

driver = criar_driver()
lote_reviews, lote_detalhes, lote_controle = [], [], []
total_reviews = sem_captura = 0

try:
    pbar = tqdm(fila.itertuples(index=False), total=len(fila), unit="perfume")
    for n, row in enumerate(pbar, start=1):
        pbar.set_description(f"{str(row.perfume)[:20]:<20}")
        try:
            detalhes, revs, total = coletar_perfume(driver, row)
            lote_detalhes.append(detalhes)
            lote_reviews.extend(revs)
            total_reviews += len(revs)
            if total and not revs:
                sem_captura += 1
                pbar.write(f"⚠ {row.perfume}: site mostra {total} resenhas, 0 capturadas (print salvo em debug_reviews/)")
            lote_controle.append({"id_perfume": row.id_perfume, "reviews_site": total,
                                  "reviews_coletadas": len(revs), "status": "ok"})
        except Exception as e:
            pbar.write(f"✘ {row.perfume}: {type(e).__name__}")
            # erro não entra no controle -> é tentado de novo na próxima execução

        if len(lote_controle) >= TAMANHO_LOTE or n == len(fila):
            salvar_tudo()
            lote_reviews, lote_detalhes, lote_controle = [], [], []

        if n % REINICIAR_DRIVER_A_CADA == 0:
            fechar(driver)
            driver = criar_driver()

        pbar.set_postfix(reviews=total_reviews, sem_captura=sem_captura)
        time.sleep(random.uniform(*PAUSA))
finally:
    salvar_tudo()
    fechar(driver)

print(f"\nReviews coletadas: {total_reviews} | perfumes com resenhas não capturadas: {sem_captura}")

In [ ]:
# ---------- 9. Resultado ----------
df_reviews = pd.read_csv(ARQUIVO_REVIEWS).drop_duplicates(["id_perfume", "review_id"])
df_detalhes = pd.read_csv(ARQUIVO_DETALHES).drop_duplicates("id_perfume", keep="last")
df_controle = pd.read_csv(ARQUIVO_CONTROLE)

print(f"Perfumes visitados: {len(df_controle)} | com detalhes: {len(df_detalhes)}")
print(f"Reviews: {len(df_reviews)} | perfumes com review: {df_reviews['id_perfume'].nunique()}\n")
print("Preenchimento dos detalhes (%):")
print((df_detalhes.notna() & (df_detalhes.astype(str) != "")).mean().mul(100).round(1).to_string())
print("\nAutor vazio nas reviews (%):", round(df_reviews["autor"].isna().mean() * 100, 1))
print("Data vazia nas reviews (%):", round(df_reviews["data_aprox"].isna().mean() * 100, 1))
df_detalhes.tail(3).T

In [ ]:
# ---------- 10. Dataset produto -> review ----------
colunas_produto = ["id_perfume", "genero", "ano", "nota_media", "votos", "reviews",
                   "reviews_coletadas", "perfumistas", "acordes",
                   "notas_topo", "notas_coracao", "notas_base", "link_perfume"]
df_produto_review = df_reviews.merge(df_detalhes[colunas_produto], on="id_perfume", how="left")
print(f"Linhas: {len(df_produto_review)}")
df_produto_review.head()